# Clustering de Estudiantes Desertores — retention = 0

Este notebook aplica K-Means **exclusivamente sobre los estudiantes que desertaron** (`retention = 0`), segmentando por modelo educativo (Pre-TEC21 / TEC21) para identificar los **perfiles de riesgo** que caracterizan a quienes no continuaron.

### Objetivo
Encontrar subgrupos homogéneos dentro de la población que desertó para:
- Detectar patrones comunes entre desertores
- Comparar perfiles de deserción Pre-TEC21 vs TEC21
- Orientar intervenciones preventivas diferenciadas

### Features
Las mismas 8 variables del análisis principal: PNA, extranjero, Prepa Tec, primera generación, extracurriculares, padres EXATEC, beca/préstamo, FTE.

| Dataset | n desertores | % del grupo |
|---------|-------------|-------------|
| Pre-TEC21 | 4,685 | 8.84% |
| TEC21 | 2,128 | 8.68% |


## 0. Imports y configuración

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.spatial.distance import cosine as cos_dist
from scipy.optimize import linear_sum_assignment

SEED   = 42
K      = 3  # K=3 para grupos más pequeños de desertores; se evalúa también K=4
DATA   = '../data/dataset_k_means.csv'
OUT    = '../data/'
palette = ['#22c55e', '#3b82f6', '#f59e0b', '#ef4444']

FEATURES = [
    'PNA', 'is_foreign', 'estuvo.prepa_tec', 'first.generation.yes',
    'has_extracurriculars', 'parents_exatec_enc',
    'total.scholarship.loan', 'FTE'
]

LABELS = {
    'PNA':                   'PNA',
    'is_foreign':            'Extranjero',
    'estuvo.prepa_tec':      'Prepa Tec',
    'first.generation.yes':  'Primera gen.',
    'has_extracurriculars':  'Extracurricular',
    'parents_exatec_enc':    'Padres EXATEC',
    'total.scholarship.loan':'Beca/Préstamo',
    'FTE':                   'Carga (FTE)',
}
print("Configuración lista ✓")


Configuración lista ✓


## 1. Carga y filtrado — solo desertores

In [2]:
df_raw = pd.read_csv(DATA)

# ── Feature engineering ──────────────────────────────────────────────────
# is_foreign: 1 si foreign_Yes: Foreigner = 1, de lo contrario 0
df_raw['is_foreign'] = df_raw['foreign_Yes: Foreigner'].astype(int)

df_raw['dropout'] = 1 - df_raw['retention']

print(f"Dataset total: n={len(df_raw):,}")
print(f"  Retenidos:   n={df_raw['retention'].sum():,}  ({df_raw['retention'].mean()*100:.2f}%)")
print(f"  Desertores:  n={(df_raw['retention']==0).sum():,}  ({df_raw['dropout'].mean()*100:.2f}%)")
print()
print(f"  Extranjeros: n={df_raw['is_foreign'].sum():,}  ({df_raw['is_foreign'].mean()*100:.2f}%)")
print(f"  Prepa Tec:   n={df_raw['estuvo.prepa_tec'].sum():,}")
print(f"  Primera gen: n={df_raw['first.generation.yes'].sum():,}")


Dataset total: n=77,517
  Retenidos:   n=70,704  (91.21%)
  Desertores:  n=6,813  (8.79%)

  Extranjeros: n=2,638  (3.40%)
  Prepa Tec:   n=36,943
  Primera gen: n=5,773


In [3]:
# Filtrar solo desertores
dropout_df = df_raw[df_raw['retention'] == 0].copy().reset_index(drop=True)

pre_do = dropout_df[dropout_df['educational.model'] == 0].copy().reset_index(drop=True)
tec_do = dropout_df[dropout_df['educational.model'] == 1].copy().reset_index(drop=True)

print(f"Total desertores: n={len(dropout_df):,}")
print(f"  Pre-TEC21: n={len(pre_do):,}  ({len(pre_do)/len(dropout_df)*100:.1f}% de desertores)")
print(f"  TEC21:     n={len(tec_do):,}  ({len(tec_do)/len(dropout_df)*100:.1f}% de desertores)")
print()
print("Distribución de features en desertores:")
print(dropout_df[FEATURES].describe().round(2))


Total desertores: n=6,813
  Pre-TEC21: n=4,685  (68.8% de desertores)
  TEC21:     n=2,128  (31.2% de desertores)

Distribución de features en desertores:
           PNA  is_foreign  estuvo.prepa_tec  first.generation.yes  \
count  6813.00     6813.00           6813.00                6813.0   
mean     85.63        0.05              0.40                   0.1   
std       6.30        0.22              0.49                   0.3   
min      63.52        0.00              0.00                   0.0   
25%      81.00        0.00              0.00                   0.0   
50%      85.30        0.00              0.00                   0.0   
75%      90.16        0.00              1.00                   0.0   
max     100.00        1.00              1.00                   1.0   

       has_extracurriculars  parents_exatec_enc  total.scholarship.loan  \
count               6813.00             6813.00                 6813.00   
mean                   0.90                0.10                 

## 2. Preprocesamiento

In [4]:
def preprocess(subset, feats=None):
    """Normaliza (z-score) las features; elimina columnas con var=0 en el subconjunto."""
    feats = feats or FEATURES
    valid = [f for f in feats if f in subset.columns and subset[f].std() > 0]
    X     = subset[valid].fillna(subset[valid].median()).values.astype(float)
    return StandardScaler().fit_transform(X), valid

def reorder_by_dropout(labels, y):
    order   = np.argsort([y[labels==k].mean() for k in range(K)])
    mapping = {old: new for new, old in enumerate(order)}
    return np.array([mapping[l] for l in labels])

def build_profile(subset, labels, valid_feats, y_out):
    """Z-score de medias por cluster + tasa de deserción y tamaño."""
    df_c  = subset[valid_feats].copy()
    means = df_c.groupby(labels).mean()
    z     = (means - df_c.mean()) / df_c.std()
    z.index.name = 'cluster'
    z['Deserción (%)'] = [y_out[labels==k].mean()*100 for k in range(K)]
    z['n']             = [int((labels==k).sum()) for k in range(K)]
    return z

def plot_heatmap(ax_h, ax_b, profile, title, pal, y_global):
    fcols  = [c for c in profile.columns if c not in ['Deserción (%)','n']]
    data   = profile[fcols].values.astype(float)
    pretty = [LABELS.get(f,f) for f in fcols]

    im = ax_h.imshow(data, cmap='RdYlGn_r', aspect='auto', vmin=-2, vmax=2)
    ax_h.set_xticks(range(len(fcols))); ax_h.set_xticklabels(pretty, rotation=40, ha='right', fontsize=9)
    ax_h.set_yticks(range(K)); ax_h.set_yticklabels([f'C{k}' for k in range(K)], fontsize=11, fontweight='bold')
    ax_h.set_title(title, fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax_h, label='z-score', fraction=0.03, pad=0.02)
    for i in range(K):
        for j in range(len(fcols)):
            v = data[i,j]
            ax_h.text(j, i, f'{v:.2f}', ha='center', va='center',
                      fontsize=8, color='white' if abs(v)>1.2 else 'black')

    vals = profile['Deserción (%)'].values
    ns   = profile['n'].values
    bars = ax_b.bar(range(K), vals, color=pal, edgecolor='white', lw=1.5)
    ax_b.axhline(y_global*100, color='black', ls='--', lw=1.3, label=f'Global {y_global*100:.1f}%')
    ax_b.set_xticks(range(K)); ax_b.set_xticklabels([f'C{k}\n(n={ns[k]:,})' for k in range(K)], fontsize=9)
    ax_b.set_ylabel('Deserción (%)'); ax_b.set_title('Tasa de deserción')
    ax_b.legend(fontsize=9)
    for bar, val in zip(bars, vals):
        ax_b.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.25,
                  f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

print("Funciones auxiliares definidas ✓")


Funciones auxiliares definidas ✓


In [5]:
X_pre_do, feat_pre_do = preprocess(pre_do)
X_tec_do, feat_tec_do = preprocess(tec_do)

# Dummy y_out para las funciones (todos son desertores, dropout=1)
y_pre_do = pre_do['dropout'].values  # todos = 1
y_tec_do = tec_do['dropout'].values

print(f"Features Pre-TEC21 desertores ({len(feat_pre_do)}): {feat_pre_do}")
print(f"Features TEC21 desertores     ({len(feat_tec_do)}): {feat_tec_do}")


Features Pre-TEC21 desertores (8): ['PNA', 'is_foreign', 'estuvo.prepa_tec', 'first.generation.yes', 'has_extracurriculars', 'parents_exatec_enc', 'total.scholarship.loan', 'FTE']
Features TEC21 desertores     (8): ['PNA', 'is_foreign', 'estuvo.prepa_tec', 'first.generation.yes', 'has_extracurriculars', 'parents_exatec_enc', 'total.scholarship.loan', 'FTE']


## 3. Selección de K para la población de desertores

In [6]:
K_RANGE = range(2, 9)

def compute_k_metrics(X, n=5000):
    idx = np.random.choice(len(X), min(n, len(X)), replace=False)
    Xs  = X[idx]
    iner, sils, dbs = [], [], []
    for k in K_RANGE:
        km  = KMeans(n_clusters=k, random_state=SEED, n_init=15).fit(X)
        iner.append(km.inertia_)
        ls  = km.predict(Xs)
        sils.append(silhouette_score(Xs, ls))
        dbs.append(davies_bouldin_score(Xs, ls))
    return iner, sils, dbs


In [7]:
print("Calculando métricas Pre-TEC21 desertores...")
i_pre_do, s_pre_do, d_pre_do = compute_k_metrics(X_pre_do, n=len(X_pre_do))
print("Calculando métricas TEC21 desertores...")
i_tec_do, s_tec_do, d_tec_do = compute_k_metrics(X_tec_do, n=len(X_tec_do))

ks = list(K_RANGE)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Selección de K — Desertores Pre-TEC21 y TEC21\n(línea roja = K elegido)',
             fontweight='bold', fontsize=13)
for row, (iner, sil, db, lbl) in enumerate([
    (i_pre_do,s_pre_do,d_pre_do,'Pre-TEC21 desertores'),
    (i_tec_do,s_tec_do,d_tec_do,'TEC21 desertores')]):
    axes[row,0].plot(ks,iner,'o-',color='#3b82f6',lw=2); axes[row,0].axvline(K,color='red',ls='--',alpha=.7,label=f'K={K}')
    axes[row,0].set_title(f'{lbl}\nInercia'); axes[row,0].legend()
    axes[row,1].plot(ks,sil,'o-',color='#22c55e',lw=2); axes[row,1].axvline(K,color='red',ls='--',alpha=.7)
    axes[row,1].set_title(f'{lbl}\nSilhouette')
    axes[row,2].plot(ks,db,'o-',color='#ef4444',lw=2); axes[row,2].axvline(K,color='red',ls='--',alpha=.7)
    axes[row,2].set_title(f'{lbl}\nDavies-Bouldin')
plt.tight_layout()
plt.savefig(OUT+'dropout_seleccion_k.png', dpi=150, bbox_inches='tight')
plt.show()


Calculando métricas Pre-TEC21 desertores...
Calculando métricas TEC21 desertores...


## 4. K-Means sobre desertores

Usamos K=3 para los desertores dado el menor tamaño de cada grupo (Pre: ~4,685, TEC: ~2,128). Se puede ajustar según las métricas del paso anterior.


In [8]:
# Ajustar K si se desea
K_DO = 3

km_pre_do = KMeans(n_clusters=K_DO, random_state=SEED, n_init=20).fit(X_pre_do)
km_tec_do = KMeans(n_clusters=K_DO, random_state=SEED, n_init=20).fit(X_tec_do)
lbl_pre_do = km_pre_do.labels_
lbl_tec_do = km_tec_do.labels_

# Ordenar clusters por PNA descendente (más académico primero) dentro de desertores
def reorder_by_pna(labels, subset):
    order   = np.argsort([-subset['PNA'].values[labels==k].mean() for k in range(K_DO)])
    mapping = {old:new for new,old in enumerate(order)}
    return np.array([mapping[l] for l in labels])

lbl_pre_do = reorder_by_pna(lbl_pre_do, pre_do)
lbl_tec_do = reorder_by_pna(lbl_tec_do, tec_do)

sil_pre_do = silhouette_score(X_pre_do, lbl_pre_do)
sil_tec_do = silhouette_score(X_tec_do, lbl_tec_do)
db_pre_do  = davies_bouldin_score(X_pre_do, lbl_pre_do)
db_tec_do  = davies_bouldin_score(X_tec_do, lbl_tec_do)

print(f"K_DO = {K_DO}")
print(f"  Pre-TEC21 desertores → Silhouette={sil_pre_do:.3f}  DB={db_pre_do:.3f}")
print(f"  TEC21     desertores → Silhouette={sil_tec_do:.3f}  DB={db_tec_do:.3f}")
print()
for k in range(K_DO):
    np_ = (lbl_pre_do==k).sum(); nt_ = (lbl_tec_do==k).sum()
    pna_p = pre_do['PNA'].values[lbl_pre_do==k].mean()
    pna_t = tec_do['PNA'].values[lbl_tec_do==k].mean()
    print(f"  C{k}: Pre n={np_:,}({np_/len(pre_do)*100:.1f}%) PNA={pna_p:.1f} | TEC n={nt_:,}({nt_/len(tec_do)*100:.1f}%) PNA={pna_t:.1f}")


K_DO = 3
  Pre-TEC21 desertores → Silhouette=0.225  DB=1.503
  TEC21     desertores → Silhouette=0.198  DB=1.770

  C0: Pre n=81(1.7%) PNA=86.5 | TEC n=102(4.8%) PNA=88.9
  C1: Pre n=2,844(60.7%) PNA=86.0 | TEC n=1,084(50.9%) PNA=87.3
  C2: Pre n=1,760(37.6%) PNA=84.3 | TEC n=942(44.3%) PNA=84.5


## 5. Perfiles de desertores

In [9]:
def build_profile_kdo(subset, labels, valid_feats, k_do):
    df_c  = subset[valid_feats].copy()
    means = df_c.groupby(labels).mean()
    z     = (means - df_c.mean()) / df_c.std()
    z.index.name = 'cluster'
    z['n'] = [int((labels==k).sum()) for k in range(k_do)]
    z['PNA (real)'] = [subset['PNA'].values[labels==k].mean() for k in range(k_do)]
    z['% del total desertores'] = [round((labels==k).sum()/len(labels)*100,1) for k in range(k_do)]
    return z

prof_pre_do = build_profile_kdo(pre_do, lbl_pre_do, feat_pre_do, K_DO)
prof_tec_do = build_profile_kdo(tec_do, lbl_tec_do, feat_tec_do, K_DO)

print("Perfiles Pre-TEC21 desertores:")
print(prof_pre_do.round(3).to_string())
print("\nPerfiles TEC21 desertores:")
print(prof_tec_do.round(3).to_string())


Perfiles Pre-TEC21 desertores:
           PNA  is_foreign  estuvo.prepa_tec  first.generation.yes  has_extracurriculars  parents_exatec_enc  total.scholarship.loan    FTE     n  PNA (real)  % del total desertores
cluster                                                                                                                                                                              
0        0.170       0.263            -0.149                 0.513                -7.538               0.316                  -0.287 -0.311    81      86.468                     1.7
1        0.101       0.138            -0.784                 0.020                 0.133              -0.046                  -0.050 -0.018  2844      86.027                    60.7
2       -0.171      -0.235             1.274                -0.056                 0.133               0.060                   0.095  0.044  1760      84.303                    37.6

Perfiles TEC21 desertores:
           PNA  is_foreign  est

## 6. Heatmap de perfiles de desertores

In [10]:
palette3 = ['#3b82f6', '#f59e0b', '#ef4444']  # azul-naranja-rojo

def plot_heatmap_kdo(ax_h, ax_b, profile, title, pal, k_do):
    fcols  = [c for c in profile.columns if c not in ['n','PNA (real)','% del total desertores']]
    data   = profile[fcols].values.astype(float)
    pretty = [LABELS.get(f,f) for f in fcols]

    im = ax_h.imshow(data, cmap='RdYlGn_r', aspect='auto', vmin=-2, vmax=2)
    ax_h.set_xticks(range(len(fcols))); ax_h.set_xticklabels(pretty, rotation=40, ha='right', fontsize=9)
    ax_h.set_yticks(range(k_do)); ax_h.set_yticklabels([f'C{k}' for k in range(k_do)], fontsize=11, fontweight='bold')
    ax_h.set_title(title, fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax_h, label='z-score', fraction=0.03, pad=0.02)
    for i in range(k_do):
        for j in range(len(fcols)):
            v = data[i,j]
            ax_h.text(j,i,f'{v:.2f}',ha='center',va='center',
                      fontsize=8,color='white' if abs(v)>1.2 else 'black')

    ns   = profile['n'].values
    pnas = profile['PNA (real)'].values
    bars = ax_b.bar(range(k_do), pnas, color=pal[:k_do], edgecolor='white', lw=1.5)
    ax_b.set_xticks(range(k_do))
    ax_b.set_xticklabels([f'C{k}\n(n={ns[k]:,})' for k in range(k_do)], fontsize=9)
    ax_b.set_ylabel('PNA promedio'); ax_b.set_title('PNA promedio por subgrupo')
    for bar,val in zip(bars,pnas):
        ax_b.text(bar.get_x()+bar.get_width()/2, bar.get_height()+.2,
                  f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Perfiles de Clusters en Desertores (K=3)\n(ordenados de mayor a menor PNA)',
             fontsize=13, fontweight='bold')
plot_heatmap_kdo(axes[0,0], axes[0,1], prof_pre_do, 'Pre-TEC21 Desertores', palette3, K_DO)
plot_heatmap_kdo(axes[1,0], axes[1,1], prof_tec_do, 'TEC21 Desertores',     palette3, K_DO)
plt.tight_layout()
plt.savefig(OUT+'dropout_perfiles_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. PCA 2D y distribución de features

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Proyección PCA 2D — Clusters de Desertores (K=3)', fontsize=13, fontweight='bold')
for ax, X, lbl, title in [
    (axes[0], X_pre_do, lbl_pre_do, 'Pre-TEC21 desertores'),
    (axes[1], X_tec_do, lbl_tec_do, 'TEC21 desertores')]:
    pca = PCA(n_components=2, random_state=SEED)
    Xp  = pca.fit_transform(X)
    for k in range(K_DO):
        m = lbl==k
        ax.scatter(Xp[m,0],Xp[m,1],c=palette3[k],alpha=.4,s=12,label=f'C{k} (n={(lbl==k).sum()})')
    ax.set_title(title,fontweight='bold')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.savefig(OUT+'dropout_pca.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Valores reales — comparativa entre subgrupos de desertores

In [12]:
disc = [
    ('PNA',                   'PNA',           75, 100),
    ('estuvo.prepa_tec',      'Prepa Tec',      0,   1),
    ('parents_exatec_enc',    'Padres EXATEC',  0,   1),
    ('is_foreign',            'Extranjero',     0,   1),
    ('first.generation.yes',  'Primera gen.',   0,   1),
    ('has_extracurriculars',  'Extracurricular',0,   1),
    ('total.scholarship.loan','Beca/Préstamo',  0,   1),
    ('FTE',                   'Carga FTE',    0.5, 1.5),
]
fig, axes = plt.subplots(2, len(disc), figsize=(24, 8))
fig.suptitle('Valores Reales en Desertores por Cluster\nPre-TEC21 (arriba) · TEC21 (abajo)',
             fontsize=13, fontweight='bold')
for ci, (feat, label, ymin, ymax) in enumerate(disc):
    for ri, (subset, labels, gname) in enumerate([
        (pre_do,lbl_pre_do,'Pre-TEC21 drop'),(tec_do,lbl_tec_do,'TEC21 drop')]):
        ax   = axes[ri,ci]
        vals = [subset[feat].values[labels==k].mean() for k in range(K_DO)]
        bars = ax.bar(range(K_DO), vals, color=palette3, edgecolor='white', lw=1)
        rng  = ymax - ymin
        ax.set_ylim(ymin-.05*rng, ymax+.2*rng)
        ax.set_xticks(range(K_DO)); ax.set_xticklabels([f'C{k}' for k in range(K_DO)], fontsize=8)
        if ci==0: ax.set_ylabel(gname, fontsize=10, fontweight='bold')
        if ri==0: ax.set_title(label, fontsize=9, fontweight='bold')
        for bar,val in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+.01*rng,
                   f'{val:.2f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig(OUT+'dropout_valores_reales.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Interpretación de los perfiles de desertores

> Los clusters se ordenan de C0 (mayor PNA, mejor perfil académico) a C2 (menor PNA, perfil más vulnerable).

### ¿Por qué estos perfiles importan?

Segmentar a los desertores nos permite identificar **qué tipo de estudiante deserta** y diseñar intervenciones diferenciadas:

- **C0 (alto PNA, deserción involuntaria):** estudiantes con buen perfil académico que desertan probablemente por razones socioeconómicas, personales o de adaptación. Intervención: apoyo psicosocial y financiero.
- **C1 (perfil medio):** grupo heterogéneo. Intervención: tutorías académicas + seguimiento activo.
- **C2 (bajo PNA, primera generación):** estudiantes con bajo capital académico e institucional. Intervención: programa de acompañamiento intensivo desde el primer semestre.


## 10. Matching desertores Pre-TEC21 ↔ TEC21

In [13]:
# Matching entre clusters de desertores
feat_sh_do = [f for f in feat_pre_do if f in feat_tec_do]

X_ps = StandardScaler().fit_transform(pre_do[feat_sh_do].fillna(pre_do[feat_sh_do].median()))
X_ts = StandardScaler().fit_transform(tec_do[feat_sh_do].fillna(tec_do[feat_sh_do].median()))

c_p = np.array([X_ps[lbl_pre_do==k].mean(axis=0) for k in range(K_DO)])
c_t = np.array([X_ts[lbl_tec_do==k].mean(axis=0) for k in range(K_DO)])

sim_do = np.zeros((K_DO, K_DO))
for i in range(K_DO):
    for j in range(K_DO):
        sim_do[i,j] = max(0.0, 1 - cos_dist(c_p[i], c_t[j]))

row_i, col_i = linear_sum_assignment(-sim_do)
match_do = [(r,c,sim_do[r,c]) for r,c in zip(row_i,col_i)]

print("Matching desertores Pre-TEC21 ↔ TEC21:")
sim_do_df = pd.DataFrame(sim_do,
    index=[f'Pre-C{k}' for k in range(K_DO)],
    columns=[f'TEC-C{k}' for k in range(K_DO)])
print(sim_do_df.round(3).to_string())
print()
for r,c,s in match_do:
    pna_p = pre_do['PNA'].values[lbl_pre_do==r].mean()
    pna_t = tec_do['PNA'].values[lbl_tec_do==c].mean()
    tag   = '🟢' if s>=.4 else ('🟡' if s>=.15 else '🔴')
    print(f"  {tag} Pre-C{r} (PNA={pna_p:.1f}) ↔ TEC-C{c} (PNA={pna_t:.1f})  sim={s:.3f}")


Matching desertores Pre-TEC21 ↔ TEC21:
        TEC-C0  TEC-C1  TEC-C2
Pre-C0   0.048   0.000    0.00
Pre-C1   0.366   0.895    0.00
Pre-C2   0.000   0.000    0.98

  🔴 Pre-C0 (PNA=86.5) ↔ TEC-C0 (PNA=88.9)  sim=0.048
  🟢 Pre-C1 (PNA=86.0) ↔ TEC-C1 (PNA=87.3)  sim=0.895
  🟢 Pre-C2 (PNA=84.3) ↔ TEC-C2 (PNA=84.5)  sim=0.980


In [14]:
# Visualización matching desertores
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax_h = axes[0]
im = ax_h.imshow(sim_do, cmap='Blues', vmin=0, vmax=max(sim_do.max()*1.1, 0.4))
ax_h.set_xticks(range(K_DO)); ax_h.set_xticklabels([f'TEC-C{k}' for k in range(K_DO)], fontsize=11)
ax_h.set_yticks(range(K_DO)); ax_h.set_yticklabels([f'Pre-C{k}' for k in range(K_DO)], fontsize=11)
ax_h.set_title('Similitud Coseno — Desertores\nPre-TEC21 ↔ TEC21', fontweight='bold', fontsize=12)
plt.colorbar(im, ax=ax_h, label='Similitud coseno', fraction=0.04)
for i in range(K_DO):
    for j in range(K_DO):
        matched = any(r==i and c==j for r,c,_ in match_do)
        ax_h.text(j,i,f'{sim_do[i,j]:.3f}',ha='center',va='center',
                 fontsize=12,fontweight='bold' if matched else 'normal')
        if matched:
            ax_h.add_patch(plt.Rectangle((j-.49,i-.49),.98,.98,fill=False,edgecolor='red',lw=3))

yp3 = list(range(K_DO-1,-1,-1))
ax_f = axes[1]
ax_f.set_xlim(0,10); ax_f.set_ylim(-.6, K_DO-.4); ax_f.axis('off')
ax_f.set_title('Matching Óptimo — Desertores', fontweight='bold', fontsize=12)
for k in range(K_DO):
    col  = palette3[k]
    n_p  = int((lbl_pre_do==k).sum()); pna_p = pre_do['PNA'].values[lbl_pre_do==k].mean()
    ax_f.add_patch(mpatches.FancyBboxPatch((.05,yp3[k]-.3),3.2,.6,
        boxstyle='round,pad=0.05',facecolor=col+'33',edgecolor=col,lw=2))
    ax_f.text(1.65,yp3[k]+.06,f'Pre-C{k}',fontsize=11,fontweight='bold',color=col,ha='center')
    ax_f.text(1.65,yp3[k]-.19,f'n={n_p:,} · PNA={pna_p:.1f}',fontsize=7.5,color='#444',ha='center')
for k in range(K_DO):
    col  = palette3[k]
    n_t  = int((lbl_tec_do==k).sum()); pna_t = tec_do['PNA'].values[lbl_tec_do==k].mean()
    ax_f.add_patch(mpatches.FancyBboxPatch((6.75,yp3[k]-.3),3.2,.6,
        boxstyle='round,pad=0.05',facecolor=col+'33',edgecolor=col,lw=2))
    ax_f.text(8.35,yp3[k]+.06,f'TEC-C{k}',fontsize=11,fontweight='bold',color=col,ha='center')
    ax_f.text(8.35,yp3[k]-.19,f'n={n_t:,} · PNA={pna_t:.1f}',fontsize=7.5,color='#444',ha='center')
for pi,(r,c,s) in enumerate(match_do):
    col = '#16a34a' if s>=.4 else ('#d97706' if s>=.15 else '#dc2626')
    ax_f.annotate('',xy=(6.7,yp3[c]),xytext=(3.3,yp3[r]),
        arrowprops=dict(arrowstyle='->',color=col,lw=1.5+4*s))
    ax_f.text(5.0,(yp3[r]+yp3[c])/2+(pi-1)*.07,f'sim={s:.3f}',
        fontsize=9,ha='center',color=col,fontweight='bold',
        bbox=dict(facecolor='white',edgecolor=col,pad=1.5,alpha=.85))

plt.tight_layout()
plt.savefig(OUT+'dropout_matching.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Exportar resultados

In [15]:
prof_pre_do['grupo'] = 'Pre-TEC21 desertores'
prof_tec_do['grupo'] = 'TEC21 desertores'
pd.concat([prof_pre_do, prof_tec_do]).to_csv(OUT+'dropout_perfiles.csv')

match_out = []
for r,c,s in match_do:
    match_out.append({'par':f'Pre-C{r}↔TEC-C{c}','pre_cluster':r,'tec_cluster':c,
        'similitud':round(s,4),
        'PNA_pre':round(pre_do['PNA'].values[lbl_pre_do==r].mean(),2),
        'PNA_tec':round(tec_do['PNA'].values[lbl_tec_do==c].mean(),2),
        'n_pre':int((lbl_pre_do==r).sum()), 'n_tec':int((lbl_tec_do==c).sum()),
        'calidad':'sólido' if s>=.4 else ('moderado' if s>=.15 else 'débil')})
pd.DataFrame(match_out).to_csv(OUT+'dropout_matching.csv', index=False)
print("✓ dropout_perfiles.csv y dropout_matching.csv guardados")


✓ dropout_perfiles.csv y dropout_matching.csv guardados
